TF-IDF relies on keyword overlap rather than semantic meaning.
This reflects limitations of traditional ATS systems, which can be improved
using contextual embeddings like BERT.

In [108]:
import os
import re
import nltk
import numpy as np
import pandas as pd

from PyPDF2 import PdfReader
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aswat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aswat\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [110]:
with open("job_description.txt", "r", encoding="utf-8") as file:
    job_description = file.read()


In [112]:
def extract_text_from_pdf(file_path):
    text = ""
    reader = PdfReader(file_path)
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted
    return text

resume_folder = "resumes/"
resumes = []

for file in os.listdir(resume_folder):
    if file.endswith(".pdf"):
        text = extract_text_from_pdf(os.path.join(resume_folder, file))
        resumes.append({"filename": file, "text": text})

resumes_df = pd.DataFrame(resumes)
resumes_df


,filename,text
0,ASWATH V AI ML.pdf,ASWATH V\nAl/ML Developer\naswathv3670@gmail.c...
1,ASWATH V Data Analyst.pdf,ASWATH V\nData Analyst\naswathv3670@gmail.com ...


In [113]:
text


'ASWATH V\nData Analyst\naswathv3670@gmail.com 6282162203 Calicut Kerala 673005 \nlinkedin.com/in/aswath-v-261080226 github.com/aswath3670 \nSUMMARY\nDetail-oriented and analytical Data Analyst with hands-on experience in data collection, cleaning, visualization, and \nstatistical analysis. Proficient in Python, SQL, Excel, and Bl tools like Power BI and Tableau. Skilled in uncovering \ninsights through exploratory data analysis and machine learning techniques. Strong foundation in database \nmanagement, data-driven decision-making, and reporting. Adept at translating business requirements into \nactionable insights to support strategic goals\nEXPERIENCE\nTrainee Software Developer (Python) - Intern, CamerinFolks, Kochi\n•Gained practical experience in Python development, focusing on backend logic.\n•Assisted in designing, coding, testing, and debugging web applications using Django \nand related frameworks.09/2023 – 02/2024\n•Collaborated with cross-functional teams to improve softwar

In [114]:
resumes_df = pd.DataFrame(resumes)
resumes_df


,filename,text
0,ASWATH V AI ML.pdf,ASWATH V\nAl/ML Developer\naswathv3670@gmail.c...
1,ASWATH V Data Analyst.pdf,ASWATH V\nData Analyst\naswathv3670@gmail.com ...


In [118]:
lemmatizer = WordNetLemmatizer()

custom_stopwords = set(stopwords.words('english')) - {
    'python','sql','machine','learning','data','analysis','nlp'
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    words = text.split()
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in custom_stopwords and len(word) > 2
    ]
    return " ".join(words)


In [120]:
resumes_df['cleaned_text'] = resumes_df['text'].apply(clean_text)
cleaned_job_description = clean_text(job_description)


In [122]:
cleaned_job_description


'hiring data analyst data scientist strong experience python panda numpy scikit learn data analysis machine learning task candidate hand experience data cleaning preprocessing exploratory data analysis eda machine learning algorithm logistic regression svm random forest natural language processing nlp idf cosine similarity data visualization using matplotlib seaborn power tableau sql database mysql data extraction analysis experience django based web application git github jupyter notebook preferred familiarity deep learning tensorflow computer vision cloud platform like aws azure plus role involves translating business requirement data driven insight building predictive model creating dashboard support decision making'

In [124]:
priority_skills = [
    "python", "machine learning", "data science", "nlp",
    "sql", "django", "tensorflow", "deep learning",
    "power bi", "tableau"
]

def boost_skills(text):
    for skill in priority_skills:
        text += (" " + skill) * 3
    return text


In [126]:
resumes_df['boosted_text'] = resumes_df['cleaned_text'].apply(boost_skills)
boosted_job_description = boost_skills(cleaned_job_description)


In [128]:
documents = resumes_df['boosted_text'].tolist()
documents.append(boosted_job_description)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=3000,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(documents)


In [130]:
job_vector = tfidf_matrix[-1]
resume_vectors = tfidf_matrix[:-1]

similarity_scores = cosine_similarity(resume_vectors, job_vector).flatten()


In [142]:
resumes_df['match_percentage'] = (similarity_scores * 100).round(2)


In [144]:
final_results = resumes_df[['filename', 'match_percentage']] \
                    .sort_values(by='match_percentage', ascending=False)

final_results


,filename,match_percentage
1,ASWATH V Data Analyst.pdf,32.06
0,ASWATH V AI ML.pdf,28.63


In [148]:
resumes_df['skills'] = resumes_df['cleaned_text'].apply(extract_skills)
resumes_df[['filename', 'skills']]


,filename,skills
0,ASWATH V AI ML.pdf,"[python, machine learning, data science, sql, ..."
1,ASWATH V Data Analyst.pdf,"[python, machine learning, data science, sql, ..."
